In [1]:
import torch
from torch import nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
splits = {'train': 'small/train-00000-of-00001.parquet', 'validation': 'small/validation-00000-of-00001.parquet', 'test': 'small/test-00000-of-00001.parquet'}
dataset = pd.read_parquet("hf://datasets/google/code_x_glue_cc_code_refinement/" + splits["train"])
dataset.head()

,id,buggy,fixed
0,0,public java.lang.String METHOD_1 ( ) { return ...,public java.lang.String METHOD_1 ( ) { return ...
1,1,public boolean METHOD_1 ( java.lang.String nam...,public boolean METHOD_1 ( java.lang.String nam...
2,2,public char METHOD_1 ( java.lang.String VAR_1 ...,public char METHOD_1 ( java.lang.String VAR_1 ...
3,3,private void METHOD_1 ( ) { VAR_1 . METHOD_2 (...,private void METHOD_1 ( ) { VAR_1 . METHOD_2 (...
4,4,public TYPE_1 METHOD_1 ( ) { java.lang.System....,public TYPE_1 METHOD_1 ( ) { return this . VAR...


In [3]:
dataset = dataset[:10000]
len(dataset)

10000

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [58]:
from torch.utils.data import Dataset,DataLoader,random_split





class CodeDataset(Dataset):
    def __init__(self,data,max_tokens):
        self.x = data["buggy"]
        self.x = list(map(self.tokensize,self.x))
        temp = np.zeros((len(self.x),max_tokens))
        for i in range(len(temp)):
            temp[i,:min(len(self.x[i]),max_tokens)] = np.array(self.x[i])
        self.x = torch.LongTensor(temp)
        self.y = data["fixed"]
        self.y = list(map(self.tokensize,self.y))
        temp = np.zeros((len(self.y),max_tokens))
        for i in range(len(temp)):
            temp[i,:min(len(self.y[i]),max_tokens)] = np.array(self.y[i])
        self.y = torch.LongTensor(temp)

    def __len__(self):
        return len(self.x)

    keywords = ['abstract', 'continue', 'for', 'new', 'switch', 'assert', 'default', 'goto', 'package', 'synchronized', 'boolean', 'do', 'if', 'private', 'this', 'break', 'double', 'implements', 'protected', 'throw', 'byte', 'else', 'import', 'public', 'throws', 'case', 'enum', 'instanceof', 'return', 'transient', 'catch', 'extends', 'int', 'short', 'try', 'char', 'final', 'interface', 'static', 'void', 'class', 'finally', 'long', 'strictfp', 'volatile', 'const', 'float', 'native', 'super', 'while','instanceof','java','lang','String','METHOD','[END]']

    operator = []


    def tokensize(self,x):
        j = -1
        tokens = []
        for i in range(len(x)):
            if x[i].isalpha():
                if j == -1:
                    j = i
            else:
                if j != -1:
                    if x[j:i] in self.keywords:
                        tokens.append(128+self.keywords.index(x[j:i]))
                    else:
                        tokens.extend(list(map(ord,x[j:i])))
                    j = -1
                if x[i] == ' ':
                    tokens.append(0)
                else:
                    tokens.append(ord(x[i]))
        tokens.append(128+len(self.keywords)-1)
        return tokens
    @staticmethod
    def getVocab():
        return 128+len(CodeDataset.keywords)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]


In [59]:
dataset_trh = CodeDataset(dataset,400)
valid_ratio = .1
test_ratio = .1

test_size = int(test_ratio*len(dataset_trh))
valid_size = int(valid_ratio*len(dataset_trh))
training_size = len(dataset_trh) - test_size - valid_size


(train_data, test_data, valid_data) = random_split(dataset_trh,[training_size,test_size,valid_size])

In [60]:
for i in range(len(dataset_trh)):
    if(len(dataset_trh[i][0])>400):
        print(len(dataset_trh[i][0]))

In [61]:
class Embedder(nn.Module):
    def __init__(self,vocab,embemding_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab,embemding_dim)
    def forward(self,x):
        return self.emb(x)

In [62]:
import torch
from torch import nn
import numpy as np


class LSTMEncoder(torch.nn.Module):
    def __init__(self,input_dim,history,cell_size,device):
        super().__init__()
        self.history = history
        self.cell_size = cell_size
        self.device = device
        self.input_dim = input_dim
        self.combined_size = self.history+input_dim
        self.W_f = nn.Sequential(
            nn.Linear(self.combined_size,self.cell_size),
            nn.Sigmoid(),
        )
        self.W_i1 = nn.Sequential(
            nn.Linear(self.combined_size,self.cell_size),
            nn.Sigmoid(),
        )
        self.W_i2 = nn.Sequential(
            nn.Linear(self.combined_size,self.cell_size),
            nn.Tanh(),
        )
        self.W_hc = nn.Sequential(
            nn.Linear(self.cell_size,self.history),
            nn.Tanh()
        )
        self.W_hh = nn.Sequential(
            nn.Linear(self.combined_size,self.history),
            nn.Sigmoid()
        )

    def forward(self,x):
        h = torch.zeros(len(x),self.history).to(self.device)
        c = torch.zeros(len(x),self.cell_size).to(self.device)
        outputs = []
        for i  in range(self.input_dim):
            combined_inputs = torch.concat((h,x[:,i,:]),dim = -1)

            # forget part
            c= c * self.W_f(combined_inputs)
            # storing info
            c= c + self.W_i1(combined_inputs)*self.W_i2(combined_inputs)
            # getting output
            h = self.W_hh(combined_inputs) * self.W_hc(c)
            outputs.append([h,c])
        return outputs

class LSTMDecoder(torch.nn.Module):
    def __init__(self,output_dim,history,cell_size,output_size,device):
        super().__init__()
        self.history = history
        self.cell_size = cell_size
        self.device = device
        self.output_dim = output_dim
        self.output_size = output_size
        self.combined_size = self.history+output_dim
        self.W_f = nn.Sequential(
            nn.Linear(self.combined_size,self.cell_size),
            nn.Sigmoid(),
        )
        self.W_i1 = nn.Sequential(
            nn.Linear(self.combined_size,self.cell_size),
            nn.Sigmoid(),
        )
        self.W_i2 = nn.Sequential(
            nn.Linear(self.combined_size,self.cell_size),
            nn.Tanh(),
        )
        self.W_hc = nn.Sequential(
            nn.Linear(self.cell_size,self.history),
            nn.Tanh()
        )
        self.W_hh = nn.Sequential(
            nn.Linear(self.combined_size,self.history),
            nn.Sigmoid()
        )
        self.W_oh = nn.Sequential(
            nn.Linear(self.history+self.cell_size+output_dim,output_dim)
        )

    def forward(self,h,c,x):
        outputs = torch.zeros((h.shape[0],self.output_size,self.output_dim)).to(device)
        for i  in range(self.output_size):
            combined_inputs = torch.concat((h,x),dim = -1)
            # forget part
            c= c * self.W_f(combined_inputs)
            # storing info
            c= c + self.W_i1(combined_inputs)*self.W_i2(combined_inputs)
            # getting output
            h = self.W_hh(combined_inputs) * self.W_hc(c)
            x = self.W_oh(torch.concat((h,c,x),dim = 1))
            outputs[:,i,:] = x
        return outputs

In [63]:
class LSTMEncoderDecoder(nn.Module):
    def __init__(self,vocab,max_tokens,embedding_dim,device):
        super().__init__()
        self.emb = Embedder(vocab,embedding_dim)
        self.encoder = LSTMEncoder(embedding_dim,128,512,device)
        self.decoder = LSTMDecoder(embedding_dim,128,512,max_tokens,device)
        self.device = device
        self.embedding_dim = embedding_dim
        self.vocab = vocab
        self.ff = nn.Sequential(
            nn.Linear(embedding_dim,max_tokens),
            nn.Linear(max_tokens,vocab),
            nn.Softmax(-1)
        )
    def forward(self,x):
        x = self.emb(x)
        h,c = self.encoder(x)[-1]
        y = self.decoder(h,c,x[:,-1,:])
        return self.ff(y)


In [64]:
train_dataset = DataLoader(train_data,128)
valid_dataset = DataLoader(valid_data,2048)
test_dataset = DataLoader(test_data,2048)
training_size

8000

In [65]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.memory_allocated()

37839360

In [69]:
import gc
gc.collect()
torch.cuda.empty_cache()

model = LSTMEncoderDecoder(CodeDataset.getVocab(),400,128,device)
model.to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

opt = torch.optim.Adam(model.parameters(),lr = 1e-4)

max_epochs = 100
model.train()
for i in range(max_epochs):
    s = 0
    for (x,y) in train_dataset:
        x = x.to(device)
        y = y.to(device)
        opt.zero_grad()
        pred = model(x).transpose(1,2)
        loss = loss_fn(pred,y)
        loss.backward()
        s+=loss.item()
        opt.step()
    if (i%5 == 0):
        with torch.no_grad():
            total = 0
            n = 0
            for (x,y) in valid_dataset:
                x = x.to(device)
                y = y.to(device).reshape(-1)
                pred = model(x)
                pred = torch.argmax(pred,dim = -1).reshape(-1)
                total += (pred == y).sum() - (0 == y).sum()
                n += len(y)- (0 == y).sum()
        print(total/n)

tensor(-4.7955, device='cuda:0')


KeyboardInterrupt: 

In [70]:
torch.save(model.state_dict(), f"./model_parameter.trhmodel")


model.eval()

total = 0
with torch.no_grad():
    for (x,y) in test_dataset:
        x = x.to(device)
        y = y.to(device).reshape(-1)
        pred = model(x)
        pred = torch.argmax(pred,dim = -1).reshape(-1)
        total += (pred == y).sum()- (0 == y).sum()
        n += len(y)- (0 == y).sum()

print(total/n)

tensor(-2.3677, device='cuda:0')


In [72]:
x = test_data[0:1][0].to(device)
y = test_data[0:1][1].to(device).reshape(-1)
pred = model(x)
print(pred.cpu())
pred = torch.argmax(pred,dim = -1).reshape(-1)
print(y.cpu(),pred.cpu())
total += (pred == y).sum()
n += len(y)

tensor([[[4.3769e-07, 3.1762e-07, 1.1233e-06,  ..., 1.8370e-07,
          8.6808e-07, 1.2988e-06],
         [1.2237e-04, 6.5062e-05, 2.2636e-04,  ..., 8.1014e-05,
          1.2375e-03, 7.9341e-04],
         [1.1665e-06, 4.1560e-07, 2.3598e-06,  ..., 7.1105e-07,
          4.6623e-05, 1.5169e-05],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00]]], grad_fn=<ToCopyBackward0>)
tensor([151,   0, 167,   0, 182,  95,  49,   0,  40,   0,  84,  89,  80,  69,
         95,  49,   0,  86,  65,  82,  95,  49,   0,  41,   0, 152,   0, 179,
         46, 105, 111,  46,  73,  79,  69, 120,  99, 101, 112, 116, 105, 111,
        110,   0,  44,   0,  84,  89,  80,  69,  95,  50,   0,  44,   0,  84,
         89,  80,  69,  95,  51,   0, 123,   0, 179

In [ ]:
class Attention(nn.Module):
    def __init__(self,size,size2):
        super().__init__()
        self.W_Q = nn.Parameter(torch.randn((size,size2)))
        self.W_K = nn.Parameter(torch.randn((size,size2)))
        self.W_V = nn.Parameter(torch.randn((size,size2)))
        self.sf = nn.Softmax(dim=-1)
        self.size2 = size2

    def forward(self,x):
        return self.sf(1/np.sqrt(self.size2)*((x@self.W_Q)@(x@self.W_V).mT))@x@self.W_V

class AddAndNormalise(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,x,y):
        h = torch.concat([x,y],dim = -1)
        return (h - h.mean())/h.std()


class MultiHeadAttention(nn.Module):
    def __init__(self,nheads,input_dim,attention_layers,device):
        super().__init__()
        self.device = device
        self.W_Q = nn.Parameter(torch.randn((input_dim,attention_layers*nheads)))
        self.W_K = nn.Parameter(torch.randn((input_dim,attention_layers*nheads)))
        self.W_V = nn.Parameter(torch.randn((input_dim,attention_layers*nheads)))
        self.W_O = nn.Parameter(torch.randn((attention_layers*nheads,attention_layers*nheads)))

        self.sf = nn.Softmax(dim=-1)
        self.attention_layers = attention_layers
        self.nheads = nheads

    def forward(self,x):
        batchsize, input_len = x.shape[0],x.shape[1]
        Q = x@self.W_Q
        K = x@self.W_K
        V = x@self.W_V

        """
        previous attempt
        Qs = torch.chunk(Q,self.nheads,dim = -1)
        Ks = torch.chunk(K,self.nheads,dim = -1)
        Vs = torch.chunk(V,self.nheads,dim = -1)
        c = torch.concat([self.scaledDotProduct(Qs[i],Ks[i],Vs[i]) for i in range(self.nheads)],dim=-1)
        return c@self.W_O
        """

        Qs = Q.reshape(batchsize,input_len,self.nheads,self.attention_layers).transpose(1,2)
        Ks = K.reshape(batchsize,input_len,self.nheads,self.attention_layers).transpose(1,2)
        Vs = V.reshape(batchsize,input_len,self.nheads,self.attention_layers).transpose(1,2)

        out = self.scaledDotProduct(Qs,Ks,Vs).transpose(1,2).reshape(batchsize,input_len,-1)
        return out@self.W_O


    def scaledDotProduct(self,Q,K,V):
        return self.sf(1/torch.sqrt(torch.Tensor([Q.shape[-1]]).to(self.device))*(Q@(K.mT)))@V


class Encoder(torch.nn.Module):
    def __init__(self,input_dim,heads,attention_dim,hidden_dim,output_dim,device):
        super().__init__()
        self.device = device
        if attention_dim % heads != 0:
            raise ArithmeticError(f"take a divisible number for heads , heads{heads} , attention_dim{attention_dim}")
        self.att = MultiHeadAttention(heads,input_dim+1,attention_dim//heads,device)
        self.norm = AddAndNormalise()
        self.W_1 = nn.Parameter(torch.randn((attention_dim+input_dim,hidden_dim)))
        self.b_1 = nn.Parameter(torch.randn(hidden_dim))
        self.W_2 = nn.Parameter(torch.randn((hidden_dim,output_dim)))
        self.b_2 = nn.Parameter(torch.randn(output_dim))
        self.r = nn.ReLU()
        self.ff = lambda x : self.r(x@self.W_1+self.b_1)@self.W_2 + self.b_2

    def forward(self,x):
        x2 = self.position_embedded(x)
        x3 = self.att(x2)
        #print(x3.shape)
        x4 = self.norm(x2,x3)

        return self.ff(x4)

    def position_embedded(self,x):
        x = x.reshape(-1,x.shape[1],1)
        P = torch.zeros_like(x)
        for i in range(x.shape[1]):
            P[:,i,0] = 2**(-i)
        return torch.concat([x,P], dim = -1)

In [ ]:
class Decoder(nn.Module):
    def __init__(self,input_dim,heads,attention_dim,hidden_dim,output_dim,device):
        super().__init__()
        self.device = device
        if attention_dim % heads != 0:
            raise ArithmeticError(f"take a divisible number for heads , heads{heads} , attention_dim{attention_dim}")
        self.att = MultiHeadAttention(heads,input_dim+1,attention_dim//heads,device)
        self.norm = AddAndNormalise()
        self.W_1 = nn.Parameter(torch.randn((attention_dim+input_dim,hidden_dim)))
        self.b_1 = nn.Parameter(torch.randn(hidden_dim))
        self.W_2 = nn.Parameter(torch.randn((hidden_dim,output_dim)))
        self.b_2 = nn.Parameter(torch.randn(output_dim))
        self.r = nn.ReLU()
        self.ff = lambda x : self.r(x@self.W_1+self.b_1)@self.W_2 + self.b_2

    def forward(self,x):
        x2 = self.position_embedded(x)
        x3 = self.att(x2)
        #print(x3.shape)
        x4 = self.norm(x2,x3)

        return self.ff(x4)

    def position_embedded(self,x):
        x = x.reshape(-1,x.shape[1],1)
        P = torch.zeros_like(x)
        for i in range(x.shape[1]):
            P[:,i,0] = 2**(-i)
        return torch.concat([x,P], dim = -1)